#Comparando MLP y CNN en Fashion-MNIST

En este notebook compararemos dos enfoques para clasificación de imágenes:

- una **red densa (MLP)**,
- y una **red neuronal convolucional (CNN)**.

## Objetivos
- Preparar los datos en dos formatos distintos.
- Entrenar un modelo **MLP** y un modelo **CNN** sobre el mismo dataset.
- Comparar su desempeño en test.
- Analizar por qué una CNN suele ser más adecuada para imágenes.
- Probar una pequeña mejora sobre la CNN base.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 1. Carga del dataset

Usaremos nuevamente **Fashion-MNIST**, para que la comparación entre modelos sea justa y fácil de interpretar.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

## 2. Preprocesamiento

Vamos a preparar los datos de dos maneras:

### Para el MLP
El modelo denso necesita imágenes vectorizadas, por lo que haremos **flatten**.

### Para la CNN
El modelo convolucional necesita imágenes con forma:

`(alto, ancho, canales)`

En este caso, como son imágenes en escala de grises, el número de canales será `1`.

In [ ]:
# Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

# Versión para MLP: flatten
X_train_mlp = X_train.reshape(len(X_train), 28 * 28)
X_test_mlp  = X_test.reshape(len(X_test), 28 * 28)

# Versión para CNN: agregar canal
X_train_cnn = np.expand_dims(X_train, axis=-1)
X_test_cnn  = np.expand_dims(X_test, axis=-1)

print("X_train_mlp shape:", X_train_mlp.shape)
print("X_test_mlp shape: ", X_test_mlp.shape)
print("X_train_cnn shape:", X_train_cnn.shape)
print("X_test_cnn shape: ", X_test_cnn.shape)

### Observación

Aquí aparece una diferencia importante:

- el **MLP** recibe cada imagen como un vector de longitud `784`,
- la **CNN** recibe cada imagen como un tensor de forma `28 x 28 x 1`.

Esta diferencia refleja dos maneras distintas de representar la información.

## 3. Modelo 1: MLP

Primero construiremos una red densa simple.

Arquitectura:
- capa de entrada implícita,
- una capa oculta `Dense`,
- capa de salida con `softmax`.

In [ ]:
mlp_model = keras.Sequential([
    layers.Input(shape=(28 * 28,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

mlp_model.summary()

### Nota sobre `sparse_categorical_crossentropy`

En problemas de **clasificación multiclase**, hay dos formas comunes de representar las etiquetas:

#### Opción 1: etiquetas como enteros
Por ejemplo:

- `0`
- `1`
- `2`
- ...
- `9`

En ese caso, usamos:

`loss="sparse_categorical_crossentropy"`

#### Opción 2: etiquetas en formato one-hot
Por ejemplo, si la clase correcta es la 2 en un problema de 4 clases:

`[0, 0, 1, 0]`

En ese caso, usamos:

`loss="categorical_crossentropy"`

---

### Entonces, ¿qué estamos haciendo aquí?

En este notebook, las etiquetas de `Fashion-MNIST` vienen como números enteros (`0, 1, ..., 9`), por ejemplo:

- `0` = T-shirt/top
- `1` = Trouser
- ...
- `9` = Ankle boot

Por eso usamos:

`loss="sparse_categorical_crossentropy"`

---

### Idea importante

Ambas funciones de pérdida sirven para **clasificación multiclase**.

La diferencia no está en el problema, sino en **cómo están codificadas las etiquetas**:

- enteros  → `sparse_categorical_crossentropy`
- one-hot  → `categorical_crossentropy`

In [ ]:
mlp_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 4. Entrenamiento del MLP

In [ ]:
history_mlp = mlp_model.fit(
    X_train_mlp,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
mlp_test_loss, mlp_test_acc = mlp_model.evaluate(X_test_mlp, y_test, verbose=0)

print("MLP - Test loss:    ", mlp_test_loss)
print("MLP - Test accuracy:", mlp_test_acc)

## 5. Modelo 2: CNN

Ahora construiremos una CNN simple sobre el mismo dataset.

Arquitectura:
- `Conv2D`
- `MaxPooling2D`
- `Flatten`
- `Dense`
- salida con `softmax`

In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 6. Entrenamiento de la CNN

In [ ]:
history_cnn = cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)

print("CNN - Test loss:    ", cnn_test_loss)
print("CNN - Test accuracy:", cnn_test_acc)

## 7. Comparación de resultados

Ahora comparemos ambos modelos en términos de:
- desempeño en test,
- representación de la entrada,
- y número de parámetros.

In [ ]:
print("Resumen de resultados:")
print(f"MLP - Test accuracy: {mlp_test_acc:.4f}")
print(f"CNN - Test accuracy: {cnn_test_acc:.4f}")

In [ ]:
comparison_names = ["MLP", "CNN"]
comparison_acc = [mlp_test_acc, cnn_test_acc]

plt.figure(figsize=(6,4))
plt.bar(comparison_names, comparison_acc)
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Comparación de accuracy en test")
plt.grid(True, axis="y")
plt.show()

### Interpretación

Aunque ambos modelos pueden aprender, la **CNN** suele rendir mejor en imágenes porque:

- conserva la estructura espacial,
- aprende patrones locales mediante filtros,
- y aprovecha mejor la información visual que una red densa básica.

## 8. Mejora simple de la CNN

Ahora probaremos una versión un poco más potente de la CNN, agregando:

- una segunda capa convolucional,
- y una capa de `Dropout`.

La idea no es construir una red compleja, sino mostrar cómo se puede mejorar gradualmente la arquitectura.

In [ ]:
cnn_model_v2 = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn_model_v2.summary()

In [ ]:
cnn_model_v2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_cnn_v2 = cnn_model_v2.fit(
    X_train_cnn,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
cnn_v2_test_loss, cnn_v2_test_acc = cnn_model_v2.evaluate(X_test_cnn, y_test, verbose=0)

print("CNN mejorada - Test loss:    ", cnn_v2_test_loss)
print("CNN mejorada - Test accuracy:", cnn_v2_test_acc)

In [ ]:
comparison_names = ["MLP", "CNN base", "CNN mejorada"]
comparison_acc = [mlp_test_acc, cnn_test_acc, cnn_v2_test_acc]

plt.figure(figsize=(7,4))
plt.bar(comparison_names, comparison_acc)
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Comparación final de accuracy en test")
plt.grid(True, axis="y")
plt.show()

## 9. Cierre

En este notebook vimos que:

- un **MLP** trata la imagen como un vector,
- una **CNN** conserva la estructura espacial,
- y una pequeña mejora en la arquitectura puede traducirse en mejor desempeño.

### Idea central
Para tareas de imágenes, una CNN suele ser una mejor primera opción que una red densa simple.

### Preguntas para reflexionar

1. ¿Por qué una CNN suele funcionar mejor que un MLP en imágenes?
2. ¿Qué papel cumple `MaxPooling2D`?
3. ¿Qué ventaja puede aportar una segunda capa convolucional?
4. ¿Por qué conviene mejorar la arquitectura gradualmente y no hacerla demasiado compleja de inmediato?

## Bonus: un ejemplo más real con `rock_paper_scissors`

Hasta ahora trabajamos con **Fashion-MNIST**, que es un dataset pequeño, ordenado y muy útil para aprender.

Ahora veremos una demo breve con un dataset más cercano a un problema real de visión computacional: **clasificar gestos de mano** en tres clases:

- `rock`
- `paper`
- `scissors`

### Idea pedagógica
La lógica general sigue siendo la misma:
- cargar imágenes,
- preprocesarlas,
- construir una CNN,
- entrenar,
- y evaluar.

Pero ahora trabajamos con imágenes RGB y con mayor variabilidad visual.

In [ ]:
# !pip install -q tensorflow-datasets

import tensorflow_datasets as tfds

### 1. Carga del dataset

Usaremos `tensorflow_datasets` (`tfds`), que permite descargar datasets listos para usar.

En este caso, `rock_paper_scissors` contiene imágenes etiquetadas de tres clases:
- rock
- paper
- scissors

In [ ]:
(ds_train_full, ds_test), ds_info = tfds.load(
    "rock_paper_scissors",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print(ds_info)

In [ ]:
class_names = ds_info.features["label"].names
print("Clases:", class_names)
print("Número de clases:", ds_info.features["label"].num_classes)

In [ ]:
train_size = ds_info.splits["train"].num_examples
test_size = ds_info.splits["test"].num_examples

print("Número de ejemplos en train original:", train_size)
print("Número de ejemplos en test original: ", test_size)

### 2. Mirar algunos ejemplos

Antes de entrenar, conviene inspeccionar algunas imágenes para entender mejor el problema.

In [ ]:
plt.figure(figsize=(10, 6))

for i, (image, label) in enumerate(ds_train_full.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()])
    plt.axis("off")

plt.tight_layout()
plt.show()

### 3. Partición y preprocesamiento

Para evitar confusiones, construiremos tres conjuntos:

- **train**: para ajustar los pesos,
- **validation**: para monitorear el entrenamiento,
- **test**: para evaluar al final.

Tomaremos el conjunto de entrenamiento original y lo dividiremos en:
- 80% para entrenamiento,
- 20% para validación.

In [ ]:
# Partición train / validation a partir del train original
val_fraction = 0.2
val_size = int(train_size * val_fraction)
train_size_final = train_size - val_size

ds_val = ds_train_full.take(val_size)
ds_train = ds_train_full.skip(val_size)

print("Ejemplos para train:", train_size_final)
print("Ejemplos para val:  ", val_size)
print("Ejemplos para test: ", test_size)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = ds_val.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
for images, labels in train_ds.take(1):
    print("Shape del batch de imágenes:", images.shape)
    print("Shape del batch de etiquetas:", labels.shape)

### Observación

Ahora cada imagen tiene forma:

`128 x 128 x 3`

Eso significa:
- altura = 128,
- ancho = 128,
- canales = 3 (RGB).

### 4. Construcción de una CNN simple

Usaremos una CNN algo más profunda que la de Fashion-MNIST, porque ahora las imágenes son más grandes y más complejas.

In [ ]:
bonus_model = keras.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(3, activation="softmax")
])

bonus_model.summary()

In [ ]:
bonus_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### 5. Entrenamiento

Como esto es una demo breve, entrenaremos solo unas pocas épocas.

La idea no es optimizar al máximo el modelo, sino mostrar cómo cambia el pipeline cuando pasamos a imágenes más reales.

In [ ]:
history_bonus = bonus_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    verbose=1
)

### 6. Evaluación final en test

Ahora sí evaluamos el modelo en el conjunto de **test**, que quedó separado del entrenamiento y de la validación.

In [ ]:
bonus_test_loss, bonus_test_acc = bonus_model.evaluate(test_ds, verbose=0)

print("Bonus - Test loss:    ", bonus_test_loss)
print("Bonus - Test accuracy:", bonus_test_acc)

### 7. Algunas predicciones

In [ ]:
for images, labels in test_ds.take(1):
    preds = bonus_model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(12, 6))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        pred_name = class_names[pred_labels[i]]
        true_name = class_names[labels[i].numpy()]
        plt.title(f"Pred: {pred_name}\nReal: {true_name}", fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    break

## Idea final del bonus

Este ejemplo muestra que la lógica de una CNN se mantiene, incluso en imágenes más realistas:

- convolución,
- pooling,
- extracción de patrones,
- clasificación final.

### Diferencia respecto a Fashion-MNIST
Ahora trabajamos con:
- imágenes RGB,
- mayor tamaño,
- mayor variabilidad visual,
- y un pipeline más cercano a aplicaciones reales.

### Conexión con la sesión 5
En problemas de imágenes reales, muchas veces conviene usar **modelos preentrenados** en lugar de entrenar desde cero.  
Eso será justamente el foco de la sesión de **transfer learning**.